In [6]:
import psycopg2
import csv
import os
from dotenv import load_dotenv
from datetime import datetime, timedelta

# Load environment variables from .env
load_dotenv()

# Database connection information
DB_USERNAME = os.getenv("DB_USERNAME")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")

def read_data_and_generate_csv(table_name, parameter_name, output_file):
    try:
        # Establish a connection to the PostgreSQL database
        conn = psycopg2.connect(
            host=DB_HOST,
            port=DB_PORT,
            database=DB_NAME,
            user=DB_USERNAME,
            password=DB_PASSWORD
        )

        # Create a cursor object
        cursor = conn.cursor()

        # Query the data from the specified table with a LIMIT of 1
        cursor.execute(f"SELECT start_date, end_date, interval, data FROM \"{table_name}\" ORDER BY start_date DESC LIMIT 1")

        # Fetch the first row
        row = cursor.fetchone()

        if row is not None:
            start_date, end_date, interval, data = row
            current_date = start_date
            interval_seconds = int(interval.total_seconds())

            # Prepare data for CSV
            csv_data = []
            for day_data in data:
                current_date += timedelta(hours=1)  # Increment by 1 hour
                for value in day_data:
                    # Check if the value is None or null and replace with 0.0
                    if value is None or value == 'null':
                        value = 0.0
                    csv_data.append([current_date, value])

            # Write data to CSV file
            with open(output_file, "w", newline="") as csv_file:
                csv_writer = csv.writer(csv_file)
                csv_writer.writerow(["Valid Date", parameter_name])  # CSV header
                csv_writer.writerows(csv_data)

            print(f"CSV file '{output_file}' created successfully.")
        else:
            print(f"No data found in table '{table_name}'.")

        # Close the cursor and the connection
        cursor.close()
        conn.close()

    except Exception as e:
        print(f"Error: {e}")

if __name__ == "__main__":
    # Specify the table name, parameter name, and output file name
    table_name = "2_metre_temperature_icond2"  # Replace with your table name
    parameter_name = "2_metre_temperature_icond2"  # Replace with your parameter name
    output_file = "./data/output.csv"  # Replace with your desired output file name

    read_data_and_generate_csv(table_name, parameter_name, output_file)


CSV file './data/output.csv' created successfully.
